In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoProcessor
from collections import Counter
from pathlib import Path

from vla.models.smolvla import smolvla 
from vla.diagnostics.self_att_llm import get_multi_layer_attention, visualize_multi_layer_heatmap
from vla.diagnostics.grad_cam import (
    grad_cam_vision_encoder,
    grad_cam_vision_encoder_multi_layer,
    grad_cam_llm_layers,
    contrastive_grad_cam,
    input_gradient_saliency,
    smoothgrad_saliency,
    visualize_grad_cam,
    visualize_multi_word_grad_cam,
    visualize_grad_cam_multi_layer,
    visualize_vision_encoder_multi_layer,
    visualize_contrastive_grid,
    visualize_saliency_comparison,
    occlusion_sensitivity,
    visualize_occlusion_multi_word,
)
from vla.utils import get_device, seed_everything
from vla.constants import PROJECT_ROOT

In [ ]:
device = get_device()
seed_everything(42)

In [ ]:
checkpoint = "HuggingFaceVLA/smolvla_libero"

episode_id = "ep0000_pick_up_the_orange_juice_and_place_it_in_the_basket"
frame_idx = 0
data_dir = PROJECT_ROOT / "data/images/libero/object"

img_path = data_dir / episode_id / f"frame{frame_idx:04d}.png"
task_path = data_dir / episode_id / "task.txt"

In [ ]:
model, model_id, action_dim = smolvla(checkpoint, device)

In [ ]:
image = Image.open(img_path).convert("RGB")

with open(task_path, "r") as f:
    task_description = f.read().strip()

In [ ]:
target_word = "juice" 

In [ ]:
attention_maps_dict, model_inputs, layer_indices, processor = get_multi_layer_attention(
    model, image, task_description, device
)

In [ ]:
visualize_multi_layer_heatmap(attention_maps_dict, model_inputs, image, layer_indices, processor, target_word)

## Grad-CAM: Dog & Cat image
Grad-CAM highlights the image regions that are *causally* important for the model's prediction of each word. Unlike attention (which is correlational), Grad-CAM uses gradients to find which vision-encoder patches most influence a token's logit.

In [ ]:
dog_cat_path = PROJECT_ROOT / "data/images/both.png"
dog_cat_image = Image.open(dog_cat_path).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(dog_cat_image)
plt.axis("off")
plt.title("Dog & Cat", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
dog_cat_description = "A dog and a cat standing together indoors"
dog_cat_words = ["dog", "cat", "standing", "indoors"]

visualize_multi_word_grad_cam(model, dog_cat_image, dog_cat_description, dog_cat_words, device)

In [ ]:
cam_dog, _, _ = grad_cam_vision_encoder(model, dog_cat_image, dog_cat_description, "dog", device)
visualize_grad_cam(cam_dog, dog_cat_image, title='Grad-CAM: "dog"')

cam_cat, _, _ = grad_cam_vision_encoder(model, dog_cat_image, dog_cat_description, "cat", device)
visualize_grad_cam(cam_cat, dog_cat_image, title='Grad-CAM: "cat"')

## Approach 1: Multi-layer Vision Encoder Grad-CAM
Hook **early → late** layers of the vision encoder (SigLIP ViT). Early layers preserve fine spatial detail, later layers have more abstract/semantic representations. If a word lights up in early layers, the model is using low-level features; if only in late layers, it's relying on high-level semantics.

In [ ]:
vit_cams_dog, vit_layers = grad_cam_vision_encoder_multi_layer(
    model, dog_cat_image, dog_cat_description, "dog", device
)
visualize_vision_encoder_multi_layer(vit_cams_dog, dog_cat_image, vit_layers, "dog")

In [ ]:
vit_cams_cat, _ = grad_cam_vision_encoder_multi_layer(
    model, dog_cat_image, dog_cat_description, "cat", device
)
visualize_vision_encoder_multi_layer(vit_cams_cat, dog_cat_image, vit_layers, "cat")

## Approach 2: LLM-layer Grad-CAM
Hook hidden states inside the **language model** layers. The image tokens flow through the LLM and get mixed with text context — by the time they reach deep layers, the model has "decided" which patches relate to which words. This should give sharper word-specific localization than the vision encoder alone.

In [ ]:
llm_cams_dog, _, llm_layers, _ = grad_cam_llm_layers(
    model, dog_cat_image, dog_cat_description, "dog", device
)
visualize_grad_cam_multi_layer(llm_cams_dog, dog_cat_image, llm_layers, "dog")

In [ ]:
llm_cams_cat, _, _, _ = grad_cam_llm_layers(
    model, dog_cat_image, dog_cat_description, "cat", device
)
visualize_grad_cam_multi_layer(llm_cams_cat, dog_cat_image, llm_layers, "cat")

## Approach 3: Occlusion Sensitivity (Gradient-Free)
The gold standard: **physically mask** each image region with gray and measure the logit drop.
No gradients, no hooks — just brute-force: which patch, when removed, hurts the prediction most?
This is slow (~64 forward passes per word) but gives the cleanest signal.
Inspired by [pytorch-grad-cam](https://github.com/jacobgil/pytorch-grad-cam) ScoreCAM and occlusion methods.

In [ ]:
visualize_occlusion_multi_word(
    model, dog_cat_image, dog_cat_description,
    ["dog", "cat"], device, grid_n=8, sigma=2.0
)

## Approach 4: More Discriminative Prompt
The prompt matters! A vague description spreads attention broadly. A prompt that explicitly mentions both objects with distinct roles forces the model to differentiate.
Compare: generic prompt vs discriminative prompt.

In [ ]:
prompt_generic = "A dog and a cat standing together indoors"
prompt_discriminative = "Describe the dog and the cat separately. The bulldog is standing behind the tabby cat."

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for row, (desc, label) in enumerate([
    (prompt_generic, "Generic prompt"),
    (prompt_discriminative, "Discriminative prompt"),
]):
    img_size = (dog_cat_image.width, dog_cat_image.height)
    extent = [0, dog_cat_image.width, dog_cat_image.height, 0]

    axes[row, 0].imshow(dog_cat_image)
    axes[row, 0].set_title(f"{label}\n(original)", fontsize=11, fontweight="bold")
    axes[row, 0].axis("off")

    for col, word in enumerate(["dog", "cat"], start=1):
        cam, _, _ = grad_cam_vision_encoder(model, dog_cat_image, desc, word, device)
        from vla.diagnostics.self_att_llm import _smooth_and_resize, _add_contour
        smooth = _smooth_and_resize(cam, img_size, sigma=1.5)
        axes[row, col].imshow(dog_cat_image)
        axes[row, col].imshow(smooth, cmap="inferno", alpha=0.6, extent=extent)
        _add_contour(axes[row, col], smooth, extent)
        axes[row, col].set_title(f'"{word}"', fontsize=11, fontweight="bold")
        axes[row, col].axis("off")

fig.suptitle("Prompt Impact on Grad-CAM Localization", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## Approach 5: Contrastive Grad-CAM
Standard Grad-CAM asks "what's important for **dog**?" — but both dog and cat share background features (floor, walls) that are generically important. **Contrastive Grad-CAM** backprops from *(logit_dog − logit_cat)*, highlighting only what's **uniquely** important for one vs the other. This should give the sharpest object discrimination.

In [ ]:
visualize_contrastive_grid(
    model, dog_cat_image, dog_cat_description,
    [("dog", "cat"), ("cat", "dog")],
    device
)

## Approach 6: Pixel-Level Saliency — Input Gradient vs SmoothGrad vs Grad-CAM
Grad-CAM is limited to the spatial resolution of the ViT patches (~16×16 grid). Two pixel-level alternatives:
- **Input×Gradient**: Gradient of the logit w.r.t. raw input pixels — full resolution, single backward pass.
- **SmoothGrad**: Average input gradients over 20 noisy copies of the image. Removes gradient noise, produces clean contours.

Both operate at the **full image resolution** instead of the coarse patch grid.

In [ ]:
visualize_saliency_comparison(
    model, dog_cat_image, dog_cat_description,
    ["dog", "cat"], device, n_smooth=20
)

## Approach 7: Finer Occlusion (14×14 grid)
The earlier 8×8 occlusion was coarse. Going to 14×14 = 196 patches per word gives much finer spatial resolution at the cost of more forward passes (~4 min per word).

In [ ]:
visualize_occlusion_multi_word(
    model, dog_cat_image, dog_cat_description,
    ["dog", "cat"], device, grid_n=14, sigma=1.5
)